# 06 — Họ láng giềng theo phiên (SKNN → STAN → VSTAN)

Bước H của pipeline. Mọi mô hình dùng **cùng** bộ query validation, cùng bộ lọc ứng viên tại `t`, cùng fallback (chọn ở notebook 05) và cùng hàm metric.

- **SKNN**: láng giềng là phiên train; cosine nhị phân giữa prefix và phiên; điểm item = tổng độ tương đồng của k phiên gần nhất chứa item.
- **STAN**: trọng số `exp(-d/λ1)` theo vị trí trong prefix, `exp(-Δt/λ2)` theo độ cũ của phiên láng giềng, `exp(-|Δpos|/λ3)` theo khoảng cách trong phiên láng giềng tới item chung gần nhất.
- **VSTAN**: STAN + trọng số IDF `log(N/df)^λidf` cho item chung (độ tương đồng) và cho item ứng viên (điểm).
- Câu hỏi nghiên cứu 1: so sánh **trọng số đều** với **trọng số theo hành vi** cho từng mô hình.
- KNN chỉ dùng 50 nhóm gần nhất của prefix (`protocol.knn.max_prefix_groups`); kho phiên gồm các phiên train có ít nhất 2 item phân biệt, mỗi item chỉ chỉ mục `sample_size` phiên gần nhất.
- Tune theo từng giai đoạn trên validation: tham số cấu trúc với trọng số đều trước, rồi trọng số hành vi với cấu trúc tốt nhất.
- Đầu ra: `reports/05_08_validation/knn_validation.json`. **Không đọc test.**

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.samples import prefix_events
from rr.models import SessionKNN, popularity
from rr.evaluation import load_samples, run, flat

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/05_08_validation"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
WIDTH = 200  # số ứng viên mô hình đưa vào bước lọc

samples_report = read_json(ROOT / "reports/04_samples/metrics.json")
metadata_report = read_json(ROOT / "reports/03_metadata/metrics.json")
split_report = read_json(ROOT / "reports/02_session_split/metrics.json")
pop_report = read_json(EVAL_DIR / "popularity_validation.json")
for name, r in [("samples", samples_report), ("popularity_validation", pop_report)]:
    require(r["protocol"]["version"] == PROTOCOL["version"], f"{name} được tạo với protocol khác.")
expected = {p: samples_report["artifacts"][p] for p in [
    "data/processed/item_vocab.parquet", "data/processed/samples/validation_queries.parquet",
    "data/processed/samples/validation_targets.parquet"]}
for p in ["data/processed/item_property_history.parquet", "data/processed/item_catalog.parquet"]:
    expected[p] = metadata_report["artifacts"][p]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| protocol", PROTOCOL["version"], "| fallback", pop_report["selected"]["fallback"])

## 2. Ví dụ biên tính tay

Kho phiên: s1 = {1, 2}, s2 = {1, 3, 4}, s3 = {5, 6}. Prefix = {1}.

- SKNN: sim(s1) = 1/√2, sim(s2) = 1/√3; điểm item 2 = 1/√2, item 3 = item 4 = 1/√3, item 1 = tổng hai giá trị.

In [ ]:
def _ev(rows):
    return pd.DataFrame(rows, columns=["session_id", "group_index", "itemid", "event", "timestamp"])
_q = pd.DataFrame({"query_id": [0], "itemid": [1], "event": ["view"], "distance": [0]})
_s = SessionKNN("sknn", k=10, sample_size=10).fit(_ev([(1, 1, 1, "view", 0), (1, 2, 2, "view", 1), (2, 1, 1, "view", 2),
                                                        (2, 2, 3, "view", 3), (2, 3, 4, "view", 4), (3, 1, 5, "view", 5), (3, 2, 6, "view", 6)]))
_it, _sc = _s.recommend(_q, [0], [100], width=5)
assert _it[0].tolist() == [1, 2, 3, 4, -1]
np.testing.assert_allclose(_sc[0, 1:4], [1 / np.sqrt(2), 1 / np.sqrt(3), 1 / np.sqrt(3)])
del _q, _s, _it, _sc
print("Đã kiểm tra ví dụ tính tay SKNN.")

## 3. Dữ liệu fit, bộ lọc, fallback và prefix validation

In [ ]:
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
train_events = events[events["split"] == "train"].reset_index(drop=True)
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
val_q, val_t = load_samples(ROOT, "validation")
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(train_events, pop_report["selected"]["fallback_weights"]))
prefix = prefix_events(val_q, events[events["split"] == "validation"], columns=("itemid", "event"), max_groups=MAX_GROUPS)
qids, qtimes = val_q["query_id"].to_numpy(), val_q["query_time_ms"].to_numpy()
require(train_events["timestamp"].max() < qtimes.min(), "Phiên láng giềng phải kết thúc trước query.")
print(f"Prefix validation (≤ {MAX_GROUPS} nhóm): {len(prefix):,} sự kiện cho {len(val_q):,} query")

UNIFORM = {"view": 1, "addtocart": 1, "transaction": 1}
WEIGHT_OPTIONS = {"uniform": UNIFORM, "w_1_2_5": {"view": 1, "addtocart": 2, "transaction": 5},
                  "w_1_5_10": {"view": 1, "addtocart": 5, "transaction": 10},
                  "pop_best": pop_report["selected"]["popularity_behavior_weights"]}
results, meta = {}, {}

def evaluate_model(name, model, fit_s, recommend):
    t0 = time.perf_counter()
    items, scores = recommend(model)
    q_s = time.perf_counter() - t0
    report, per_q, topk, _ = run(items, scores, val_q, val_t, flt, fb_items, fb_scores)
    require(report["short_lists"] == 0, f"{name}: có query thiếu item.")
    report.update(fit_seconds=round(fit_s, 2), query_seconds_total=round(q_s, 2),
                  query_ms_per_query=round(1000 * q_s / len(val_q), 4),
                  model_candidates_per_query=float((items != -1).sum(axis=1).mean()))
    results[name] = report
    print(f"{name:<45} ndcg@20={report['all']['ndcg@20']:.4f} hit@20={report['all']['hit@20']:.4f} "
          f"fallback_slots={report['fallback']['slot_rate']:.3f} fit={fit_s:.0f}s query={q_s:.0f}s")
    return items, scores

## 4. SKNN, STAN và VSTAN

Giai đoạn 1 (trọng số đều): `k ∈ {50, 100, 200}`, `sample_size ∈ {500, 1000}` cho SKNN; STAN thêm `λ1 ∈ {1, 3}`, `λ2 ∈ {1, 7} ngày`, `λ3 ∈ {1, 3}` với `k`, `sample_size` tốt nhất của SKNN. Giai đoạn 2: trọng số hành vi với cấu trúc tốt nhất.

In [ ]:
def run_sknn(name, **params):
    t0 = time.perf_counter()
    model = SessionKNN(**params).fit(train_events)
    fit_s = time.perf_counter() - t0
    evaluate_model(name, model, fit_s, lambda mdl: mdl.recommend(prefix, qids, qtimes, WIDTH))
    return model

DAY = 86_400_000
structures = {}
for k in [50, 100, 200]:
    for sample in [500, 1000]:
        run_sknn(f"sknn_k{k}_s{sample}_uniform", variant="sknn", k=k, sample_size=sample)
best = max((n for n in results if n.startswith("sknn_")), key=lambda n: results[n]["all"]["ndcg@20"])
structures["sknn"] = {"k": int(best.split("_k")[1].split("_")[0]), "sample_size": int(best.split("_s")[1].split("_")[0])}
base = structures["sknn"]
for l1 in [1.0, 3.0]:
    for l2 in [1, 7]:
        for l3 in [1.0, 3.0]:
            run_sknn(f"stan_k{base['k']}_s{base['sample_size']}_l1{l1}_l2{l2}d_l3{l3}_uniform", variant="stan",
                     k=base["k"], sample_size=base["sample_size"], lambda1=l1, lambda2_ms=l2 * DAY, lambda3=l3)
best_stan = max((n for n in results if n.startswith("stan_")), key=lambda n: results[n]["all"]["ndcg@20"])
l1 = float(best_stan.split("_l1")[1].split("_")[0]); l2 = int(best_stan.split("_l2")[1].split("d")[0])
l3 = float(best_stan.split("_l3")[1].split("_")[0])
structures["stan"] = {**base, "lambda1": l1, "lambda2_ms": l2 * DAY, "lambda3": l3}
for lidf in [0.5, 1.0, 2.0]:
    run_sknn(f"vstan_k{base['k']}_s{base['sample_size']}_idf{lidf}_uniform", variant="vstan", lambda_idf=lidf,
             **{k: v for k, v in structures["stan"].items()})
best_vstan = max((n for n in results if n.startswith("vstan_")), key=lambda n: results[n]["all"]["ndcg@20"])
structures["vstan"] = {**structures["stan"], "lambda_idf": float(best_vstan.split("_idf")[1].split("_")[0])}
for variant, params in structures.items():
    for wname in [w for w in WEIGHT_OPTIONS if w != "uniform"]:
        run_sknn(f"{variant}_best_{wname}", variant=variant, weights=WEIGHT_OPTIONS[wname], **params)
meta.update({v: {"structure": s} for v, s in structures.items()})

## 5. Chọn cấu hình tốt nhất mỗi mô hình và câu hỏi nghiên cứu 1

Với mỗi mô hình: so sánh cấu hình tốt nhất khi dùng trọng số đều với cấu hình tốt nhất khi dùng trọng số theo hành vi (cùng cấu trúc).

In [ ]:
def family(name):
    return name.split("_")[0]
def weight_of(name):
    return next(w for w in WEIGHT_OPTIONS if name.endswith("_" + w))
rq1 = {}
for fam in ["sknn", "stan", "vstan"]:
    names = [n for n in results if family(n) == fam]
    uni = max((n for n in names if weight_of(n) == "uniform"), key=lambda n: results[n]["all"]["ndcg@20"])
    beh = max((n for n in names if weight_of(n) != "uniform"), key=lambda n: results[n]["all"]["ndcg@20"])
    best = max([uni, beh], key=lambda n: results[n]["all"]["ndcg@20"])
    meta[fam].update(best_run=best, weights=WEIGHT_OPTIONS[weight_of(best)], weight_name=weight_of(best))
    rq1[fam] = {"uniform_run": uni, "behavior_run": beh,
                **{f"{p}_{m}": results[n][p]["ndcg@20"] for m, n in [("uniform", uni), ("behavior", beh)]
                   for p in ["all", "strong_behavior_targets", "view_only_targets"]}}
display(pd.DataFrame(rq1).T)
best_runs = {fam: meta[fam]["best_run"] for fam in meta}
display(flat({f: results[n] for f, n in best_runs.items()}).style.format("{:.4f}"))

## 6. Theo độ dài prefix và lưu báo cáo

In [ ]:
display(pd.DataFrame({f: {b: v["ndcg@20"] for b, v in results[n]["by_prefix_len"].items()} for f, n in best_runs.items()}).style.format("{:.4f}"))
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
write_json(EVAL_DIR / "knn_validation.json", {
    "generated_at_utc": utc_now(), **environment(), "notebook": "notebooks/06_sessionknn.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs, "split": "validation", "fit_data": "train", "n_queries": len(val_q),
    "fallback": pop_report["selected"]["fallback"], "candidate_width": WIDTH, "max_prefix_groups": MAX_GROUPS,
    "weight_options": WEIGHT_OPTIONS, "runs": results, "selected": meta, "rq1_uniform_vs_behavior": rq1,
    "status": "PASS"})
print("Báo cáo:", rel(EVAL_DIR / "knn_validation.json", ROOT))